In [2]:
 
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
from scipy.stats import spearmanr, linregress
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                     BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')
 
TODAY = datetime.today().strftime('%Y-%m-%d')
print(f"Running as of: {TODAY}")

Running as of: 2026-03-24


In [23]:
"""
Real-Time Price + Technical Analysis LSTM
──────────────────────────────────────────────────────────────────────────────
Changes from price_rank_lstm.py:
  1. Fetches price data up to TODAY (not a fixed end date)
  2. Predicts using the most recent 52 weeks per ticker (no quarter boundary)
  3. Adds full technical analysis features:
       - Trend    : EMAs, MACD, ADX
       - Momentum : RSI, Stochastic, Rate of Change
       - Volume   : OBV, VWAP proxy, volume surge
       - Volatility: Bollinger Bands, ATR
       - Rank     : weekly cross-sectional rank (vs universe)
       - Predicted rank lags from prior model
"""

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
from scipy.stats import spearmanr, linregress
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                     BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

TODAY = datetime.today().strftime('%Y-%m-%d')
print(f"Running as of: {TODAY}")


# ─────────────────────────────────────────────
# STEP 1: FETCH PRICES UP TO TODAY
# ─────────────────────────────────────────────
def fetch_prices_to_today(tickers: list,
                           start:   str  = "2018-01-01",
                           end:     str  = TODAY) -> pd.DataFrame:
    """
    Fetch weekly OHLCV up to today's date.
    end defaults to today so every run gets the freshest data.
    """
    print(f"Fetching weekly prices: {start} → {end}")
    all_prices, failed = [], []

    for ticker in tickers:
        try:
            data = yf.download(ticker, start=start, end=end,
                               interval="1wk", auto_adjust=True,
                               progress=False)
            if len(data) == 0:
                failed.append(ticker)
                continue
            data = data[["Open","High","Low","Close","Volume"]].copy()
            data.columns = ["open","high","low","close","volume"]
            data["ticker"] = ticker
            data.index.name = "date"
            all_prices.append(data.reset_index())
        except Exception as e:
            failed.append(ticker)

    if failed:
        print(f"  Failed: {failed}")

    prices = pd.concat(all_prices, ignore_index=True)
    prices["date"] = pd.to_datetime(prices["date"])
    prices = prices.sort_values(["ticker","date"]).reset_index(drop=True)
    print(f"  Done: {prices['ticker'].nunique()} tickers | "
          f"{len(prices):,} rows | latest: {prices['date'].max().date()}")
    return prices


# ─────────────────────────────────────────────
# STEP 2: FULL TECHNICAL ANALYSIS FEATURES
# ─────────────────────────────────────────────
def compute_technical_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Compute full suite of technical indicators per ticker.
    All use only past data — PIT safe.
    """
    prices = prices.copy().sort_values(["ticker","date"])
    g  = prices.groupby("ticker")

    # ── Returns ───────────────────────────────────────────────────────
    prices["ret_1w"]  = g["close"].pct_change(1)
    prices["ret_2w"]  = g["close"].pct_change(2)
    prices["ret_4w"]  = g["close"].pct_change(4)
    prices["ret_8w"]  = g["close"].pct_change(8)
    prices["ret_13w"] = g["close"].pct_change(13)
    prices["ret_26w"] = g["close"].pct_change(26)
    prices["ret_52w"] = g["close"].pct_change(52)

    # Skip-month momentum
    prices["mom_12m_skip1m"] = (
        g["close"].transform(lambda x: x.shift(4) / x.shift(56) - 1)
    )

    # ── Trend: EMAs ───────────────────────────────────────────────────
    for span in [8, 13, 26, 52]:
        prices[f"ema_{span}w"] = (
            g["close"].transform(lambda x: x.ewm(span=span, adjust=False).mean())
        )

    prices["ema_cross_8_26"]  = prices["ema_8w"]  / (prices["ema_26w"]  + 1e-9) - 1
    prices["ema_cross_13_52"] = prices["ema_13w"] / (prices["ema_52w"] + 1e-9) - 1
    prices["price_vs_ema26"]  = prices["close"]   / (prices["ema_26w"]  + 1e-9) - 1
    prices["price_vs_ema52"]  = prices["close"]   / (prices["ema_52w"]  + 1e-9) - 1

    # ── MACD (12w fast, 26w slow, 9w signal) ─────────────────────────
    ema12 = g["close"].transform(lambda x: x.ewm(span=12, adjust=False).mean())
    ema26 = g["close"].transform(lambda x: x.ewm(span=26, adjust=False).mean())
    prices["macd"]        = ema12 - ema26
    prices["macd_signal"] = g["macd"].transform(
        lambda x: x.ewm(span=9, adjust=False).mean()
    )
    prices["macd_hist"]   = prices["macd"] - prices["macd_signal"]
    prices["macd_cross"]  = (prices["macd"] > prices["macd_signal"]).astype(float)

    # ── RSI (14w) ─────────────────────────────────────────────────────
    def compute_rsi(series, window=14):
        delta = series.diff()
        gain  = delta.clip(lower=0)
        loss  = (-delta).clip(lower=0)
        avg_gain = gain.ewm(com=window-1, adjust=False).mean()
        avg_loss = loss.ewm(com=window-1, adjust=False).mean()
        rs  = avg_gain / (avg_loss + 1e-9)
        rsi = 100 - (100 / (1 + rs))
        return rsi / 100   # normalize to 0→1

    prices["rsi_14w"] = g["close"].transform(compute_rsi)

    # RSI divergence: price making new high but RSI not (bearish signal)
    prices["rsi_divergence"] = (
        (g["close"].transform(lambda x: (x == x.rolling(13).max()).astype(float))) -
        (g["rsi_14w"].transform(lambda x: (x == x.rolling(13).max()).astype(float)))
    )

    # ── Stochastic %K (14w) ───────────────────────────────────────────
    low_14  = g["low"].transform(lambda x: x.rolling(14).min())
    high_14 = g["high"].transform(lambda x: x.rolling(14).max())
    prices["stoch_k"] = (prices["close"] - low_14) / (high_14 - low_14 + 1e-9)
    prices["stoch_d"] = g["stoch_k"].transform(lambda x: x.rolling(3).mean())

    # ── Rate of Change ────────────────────────────────────────────────
    prices["roc_4w"]  = g["close"].transform(lambda x: x.diff(4)  / (x.shift(4)  + 1e-9))
    prices["roc_13w"] = g["close"].transform(lambda x: x.diff(13) / (x.shift(13) + 1e-9))

    # ── Volatility ────────────────────────────────────────────────────
    prices["vol_4w"]  = g["close"].transform(lambda x: x.pct_change().rolling(4).std())
    prices["vol_13w"] = g["close"].transform(lambda x: x.pct_change().rolling(13).std())
    prices["vol_26w"] = g["close"].transform(lambda x: x.pct_change().rolling(26).std())
    prices["vol_52w"] = g["close"].transform(lambda x: x.pct_change().rolling(52).std())

    # Vol regime: is current vol high vs its own history?
    prices["vol_regime"] = prices["vol_4w"] / (prices["vol_52w"] + 1e-9)

    # ── Bollinger Bands (20w, 2 std) ──────────────────────────────────
    bb_mid   = g["close"].transform(lambda x: x.rolling(20).mean())
    bb_std   = g["close"].transform(lambda x: x.rolling(20).std())
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    prices["bb_position"] = (prices["close"] - bb_lower) / (bb_upper - bb_lower + 1e-9)
    prices["bb_width"]    = (bb_upper - bb_lower) / (bb_mid + 1e-9)

    # ── ATR (Average True Range, 14w) ─────────────────────────────────
    prev_close = g["close"].transform(lambda x: x.shift(1))
    tr = pd.concat([
        prices["high"] - prices["low"],
        (prices["high"] - prev_close).abs(),
        (prices["low"]  - prev_close).abs()
    ], axis=1).max(axis=1)
    prices["atr_14w"]      = g["ticker"].transform(lambda x: tr.loc[x.index].ewm(span=14, adjust=False).mean())
    prices["atr_pct"]      = prices["atr_14w"] / (prices["close"] + 1e-9)

    # ── Sharpe-like ───────────────────────────────────────────────────
    prices["sharpe_13w"] = prices["ret_13w"] / (prices["vol_13w"] + 1e-9)
    prices["sharpe_52w"] = prices["ret_52w"] / (prices["vol_52w"] + 1e-9)

    # ── Drawdown ──────────────────────────────────────────────────────
    prices["high_52w"] = g["high"].transform(lambda x: x.rolling(52).max())
    prices["drawdown"] = prices["close"] / (prices["high_52w"] + 1e-9) - 1

    prices["low_52w"]  = g["low"].transform(lambda x: x.rolling(52).min())
    prices["pos_52w_range"] = (
        (prices["close"] - prices["low_52w"]) /
        (prices["high_52w"] - prices["low_52w"] + 1e-9)
    )  # 0=near 52w low, 1=near 52w high

    # ── Moving averages ───────────────────────────────────────────────
    prices["ma_13w"]          = g["close"].transform(lambda x: x.rolling(13).mean())
    prices["ma_52w"]          = g["close"].transform(lambda x: x.rolling(52).mean())
    prices["price_vs_52w_ma"] = prices["close"] / (prices["ma_52w"] + 1e-9) - 1

    # ── Volume features ───────────────────────────────────────────────
    prices["vol_ratio"]  = (
        g["volume"].transform(lambda x: x.rolling(4).mean() /
                              (x.rolling(52).mean() + 1))
    )
    prices["vol_surge"]  = (prices["vol_ratio"] > 2.0).astype(float)

    # OBV (On-Balance Volume) — normalized
    def obv(grp):
        direction = np.sign(grp["close"].diff().fillna(0))
        obv_raw   = (direction * grp["volume"]).cumsum()
        obv_norm  = obv_raw / (obv_raw.abs().rolling(52).max() + 1)
        return obv_norm

    prices["obv_norm"] = g.apply(obv).reset_index(level=0, drop=True)

    # VWAP proxy (weekly close * volume / rolling volume sum)
    vwap = g.apply( lambda x: (x["close"] * x["volume"]).rolling(4).sum() /
                      (x["volume"].rolling(4).sum() + 1) ).reset_index(level=0, drop=True)

    prices["vwap_ratio"] = vwap / (prices["close"] + 1e-9)

    print(f"  Technical features computed: {len([c for c in prices.columns if c not in ['ticker','date','open','high','low','close','volume','quarter']])} indicators")
    return prices


# ─────────────────────────────────────────────
# STEP 3: WEEKLY CROSS-SECTIONAL RANK
# ─────────────────────────────────────────────
def add_weekly_rank(prices: pd.DataFrame) -> pd.DataFrame:
    prices = prices.copy().sort_values(["date","ticker"])

    prices["weekly_rank_pct"] = (
        prices.groupby("date")["ret_4w"]
        .transform(lambda x: x.rank(pct=True, na_option="keep"))
    )

    prices["rsi_rank_pct"] = (
        prices.groupby("date")["rsi_14w"]
        .transform(lambda x: x.rank(pct=True, na_option="keep"))
    )

    prices["mom_rank_pct"] = (
        prices.groupby("date")["ret_52w"]
        .transform(lambda x: x.rank(pct=True, na_option="keep"))
    )

    prices = prices.sort_values(["ticker","date"])
    prices["weekly_rank_ma4"] = (
        prices.groupby("ticker")["weekly_rank_pct"]
        .transform(lambda x: x.rolling(4, min_periods=2).mean())
    )

    def slope_8w(x):
        result = []
        vals = x.values
        for i in range(len(vals)):
            window = vals[max(0, i-7):i+1]
            if len(window) < 3 or np.isnan(window).any():
                result.append(np.nan)
            else:
                s, *_ = linregress(range(len(window)), window)
                result.append(s)
        return pd.Series(result, index=x.index)

    prices["weekly_rank_trend"] = (
        prices.groupby("ticker")["weekly_rank_pct"].transform(slope_8w)
    )
    return prices


# ─────────────────────────────────────────────
# STEP 4: ATTACH PREDICTED RANK LAGS
# ─────────────────────────────────────────────
def attach_predicted_rank(prices:      pd.DataFrame,
                           all_pred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Attach quarterly predicted rank to every week in that quarter,
    plus lag features showing prediction history.
    """
    prices = prices.copy()
    prices["quarter"] = prices["date"].dt.to_period("Q").astype(str)

    all_pred_df = all_pred_df.copy()
    all_pred_df["quarter"] = all_pred_df["quarter"].astype(str)

    qrank = (all_pred_df[["ticker","quarter","pred_rank"]]
             .sort_values(["ticker","quarter"]))

    for lag in range(1, 5):
        qrank[f"pred_rank_lag{lag}"] = (
            qrank.groupby("ticker")["pred_rank"].shift(lag)
        )

    def pred_slope(x):
        result = []
        vals = x.values
        for i in range(len(vals)):
            window = vals[max(0, i-3):i+1]
            if len(window) < 2 or np.isnan(window).any():
                result.append(np.nan)
            else:
                s, *_ = linregress(range(len(window)), window)
                result.append(s)
        return pd.Series(result, index=x.index)

    qrank["pred_rank_trend"] = (
        qrank.groupby("ticker")["pred_rank"].transform(pred_slope)
    )
    qrank["pred_rank_consistency"] = (
        qrank.groupby("ticker")["pred_rank"]
        .transform(lambda x: x.shift(1).rolling(4, min_periods=2)
                   .apply(lambda w: (w >= 0.5).mean()))
    )

    lag_cols = ["pred_rank","pred_rank_lag1","pred_rank_lag2",
                "pred_rank_lag3","pred_rank_lag4",
                "pred_rank_trend","pred_rank_consistency"]

    prices = prices.merge(
        qrank[["ticker","quarter"] + lag_cols],
        on=["ticker","quarter"], how="left"
    )
    return prices


# ─────────────────────────────────────────────
# STEP 5: PREDICT ON TODAY'S DATA
# ─────────────────────────────────────────────
def predict_today(prices:        pd.DataFrame,
                  feature_cols:  list,
                  models:        list,
                  seq_len:       int = 52) -> pd.DataFrame:
    """
    For each ticker take the MOST RECENT 52 weeks up to today
    and run the trained LSTM ensemble — no quarter boundary needed.
    """
    today     = pd.Timestamp.today()
    X_list, meta_list = [], []

    for ticker, grp in prices.groupby("ticker"):
        grp = grp.sort_values("date").reset_index(drop=True)

        # Most recent 52 weeks up to today
        available = grp[grp["date"] <= today]
        if len(available) < seq_len // 2:
            continue

        window    = available.tail(seq_len)[feature_cols]
        price_arr = window.values.astype(np.float32)

        # Pad if shorter than seq_len
        if len(price_arr) < seq_len:
            pad = np.zeros((seq_len - len(price_arr),
                            len(feature_cols)), dtype=np.float32)
            price_arr = np.vstack([pad, price_arr])

        price_arr = np.nan_to_num(price_arr, nan=0.0)
        X_list.append(price_arr)
        meta_list.append({
            "ticker":    ticker,
            "as_of":     available["date"].iloc[-1].date(),
            "data_weeks": len(available)
        })

    X    = np.array(X_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)

    # Ensemble prediction
    preds = np.mean(
        [m.predict(X, verbose=0).flatten() for m in models],
        axis=0
    )

    meta["raw_score"]   = preds
    meta["rank_pct"]    = meta["raw_score"].rank(pct=True)
    meta["rank_int"]    = meta["raw_score"].rank(ascending=False,
                                                  method="min").astype(int)
    meta["top20pct"]    = (meta["rank_pct"] >= 0.80).astype(int)
    meta["signal"]      = meta["rank_pct"].apply(
        lambda x: "STRONG BUY" if x >= 0.90 else
                  "BUY"        if x >= 0.80 else
                  "WATCH"      if x >= 0.65 else
                  "NEUTRAL"    if x >= 0.35 else
                  "AVOID"
    )

    return meta.sort_values("rank_int").reset_index(drop=True)


# ─────────────────────────────────────────────
# STEP 6: TECHNICAL ANALYSIS SUMMARY PER STOCK
# ─────────────────────────────────────────────
def technical_summary(prices: pd.DataFrame, tickers: list) -> pd.DataFrame:
    """
    For each ticker produce a human-readable technical analysis
    snapshot using the most recent week's indicators.
    """
    rows = []
    for ticker in tickers:
        grp  = prices[prices["ticker"] == ticker].sort_values("date")
        if len(grp) == 0:
            continue
        last = grp.iloc[-1]

        def sig(val, low, high, rev=False):
            if pd.isna(val):
                return "N/A"
            if not rev:
                return "Bullish" if val > high else "Bearish" if val < low else "Neutral"
            else:
                return "Bearish" if val > high else "Bullish" if val < low else "Neutral"

        rows.append({
            "ticker"         : ticker,
            "as_of"          : grp["date"].iloc[-1].date(),
            # Trend
            "ema_cross"      : "Bullish" if last.get("ema_cross_8_26", 0) > 0 else "Bearish",
            "price_vs_ma52"  : f"{last.get('price_vs_52w_ma', 0):+.1%}",
            "macd_signal"    : "Bullish" if last.get("macd_hist", 0) > 0 else "Bearish",
            # Momentum
            "rsi_14w"        : f"{last.get('rsi_14w', 0.5)*100:.0f}",
            "rsi_signal"     : sig(last.get("rsi_14w"), 0.30, 0.70),
            "stoch_k"        : f"{last.get('stoch_k', 0.5)*100:.0f}",
            "mom_13w"        : f"{last.get('ret_13w', 0):+.1%}",
            "mom_52w"        : f"{last.get('ret_52w', 0):+.1%}",
            # Volatility
            "bb_position"    : f"{last.get('bb_position', 0.5):.2f}",
            "vol_regime"     : "High" if last.get("vol_regime", 1) > 1.5 else
                               "Low"  if last.get("vol_regime", 1) < 0.7 else "Normal",
            "drawdown"       : f"{last.get('drawdown', 0):+.1%}",
            "pos_52w_range"  : f"{last.get('pos_52w_range', 0.5):.0%}",
            # Volume
            "vol_surge"      : "Yes" if last.get("vol_surge", 0) == 1 else "No",
            "obv_trend"      : "Rising" if last.get("obv_norm", 0) > 0 else "Falling",
            # Rank
            "weekly_rank"    : f"{last.get('weekly_rank_pct', 0.5):.0%}",
            "rank_trend"     : "Improving" if last.get("weekly_rank_trend", 0) > 0.01
                               else "Falling" if last.get("weekly_rank_trend", 0) < -0.01
                               else "Stable",
        })

    return pd.DataFrame(rows)


Running as of: 2026-03-24


In [36]:

def build_labeled_sequences(prices:       pd.DataFrame,
                             labeled_df:   pd.DataFrame,
                             feature_cols: list,
                             seq_len:      int = 52) -> tuple:
    """
    Build sequences at each quarter-end week for labeled quarters only.
    Target = actual cross-sectional return rank (0→1).
 
    labeled_df must have:
      - ticker, fiscalDateEnding, quarter, actual_return_rank
    """
    prices = prices.copy().sort_values(["ticker","date"])
 
    # Mark quarter-end rows
    prices["quarter"] = prices["date"].dt.to_period("Q").astype(str)
    prices["quarter_end"] = (
        prices["date"].dt.to_period("Q") !=
        prices["date"].shift(-1).dt.to_period("Q")
    )
 
    # Build (ticker, quarter) → actual_return_rank lookup
    labeled_df = labeled_df.copy()
    labeled_df["quarter"] = labeled_df["quarter"].astype(str)
    labeled_df["actual_return_rank"] = (
        labeled_df.groupby("quarter")["future_return_1y"]
        .transform(lambda x: x.rank(pct=True))
    )
    rank_map = (labeled_df[["ticker","quarter","actual_return_rank"]]
                .drop_duplicates()
                .set_index(["ticker","quarter"])["actual_return_rank"]
                .to_dict())
 
    X_list, y_list, meta_list = [], [], []
    skipped = {"no_label": 0, "short": 0, "nan": 0}
 
    for ticker, grp in prices.groupby("ticker"):
        grp     = grp.sort_values("date").reset_index(drop=True)
        qend_idx = grp[grp["quarter_end"]].index.tolist()
 
        for idx in qend_idx:
            if idx < seq_len - 1:
                skipped["short"] += 1
                continue
 
            quarter = grp.loc[idx, "quarter"]
            label   = rank_map.get((ticker, quarter), np.nan)
 
            if np.isnan(label):
                skipped["no_label"] += 1
                continue
 
            seq = grp.loc[idx - seq_len + 1 : idx, feature_cols].values.astype(np.float32)
 
            if np.isnan(seq).any():
                seq = np.nan_to_num(seq, nan=0.0)
 
            X_list.append(seq)
            y_list.append(float(label))
            meta_list.append({"ticker": ticker, "quarter": quarter})
 
    X    = np.array(X_list, dtype=np.float32)
    y    = np.array(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)
 
    print(f"\nLabeled sequences built:")
    print(f"  X shape  : {X.shape}  (samples × {seq_len} weeks × {len(feature_cols)} features)")
    print(f"  y shape  : {y.shape}")
    print(f"  Skipped  : {skipped}")
    print(f"  Quarters : {sorted(meta['quarter'].unique())}")
 
    return X, y, meta
 
 
# ─────────────────────────────────────────────
# STEP 2: BUILD TODAY'S SEQUENCES (no label)
# ─────────────────────────────────────────────
def build_today_sequences(prices:       pd.DataFrame,
                           feature_cols: list,
                           seq_len:      int = 52) -> tuple:
    """
    For each ticker take the most recent seq_len weeks up to today.
    No label needed — used for live prediction.
    """
    today  = pd.Timestamp.today()
    X_list, meta_list = [], []
 
    for ticker, grp in prices.groupby("ticker"):
        grp       = grp.sort_values("date").reset_index(drop=True)
        available = grp[grp["date"] <= today]
 
        if len(available) < seq_len // 2:
            continue
 
        window    = available.tail(seq_len)[feature_cols]
        price_arr = window.values.astype(np.float32)
 
        if len(price_arr) < seq_len:
            pad = np.zeros((seq_len - len(price_arr),
                            len(feature_cols)), dtype=np.float32)
            price_arr = np.vstack([pad, price_arr])
 
        price_arr = np.nan_to_num(price_arr, nan=0.0)
        X_list.append(price_arr)
        meta_list.append({
            "ticker":  ticker,
            "as_of":   available["date"].iloc[-1].date(),
            "quarter": available["date"].iloc[-1].to_period("Q").strftime("%YQ%q")
        })
 
    X    = np.array(X_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)
    print(f"\nToday sequences: {X.shape}  ({len(meta)} tickers as of today)")
    return X, meta
 
 
# ─────────────────────────────────────────────
# STEP 3: LSTM MODEL (sized for new feature set)
# ─────────────────────────────────────────────
def build_model(seq_len: int, n_features: int) -> tf.keras.Model:
    """
    Slightly larger than before — new feature set has ~45 features vs ~22.
    Still kept small given 90-stock universe.
    """
    inp = Input(shape=(seq_len, n_features), name="input")
    x   = LSTM(64, return_sequences=True, name="lstm_1")(inp)
    x   = Dropout(0.35)(x)
    x   = LSTM(32, return_sequences=True, name="lstm_2")(x)
    x   = Dropout(0.35)(x)
    x   = LSTM(16, name="lstm_3")(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.35)(x)
    x   = Dense(32, activation="relu", name="dense_1")(x)
    x   = Dropout(0.3)(x)
    x   = Dense(16, activation="relu", name="dense_2")(x)
    out = Dense(1,  activation="linear", name="output")(x)
 
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=1e-3), loss="mse")
    return model
 
 
# ─────────────────────────────────────────────
# STEP 4: WALK-FORWARD RETRAIN
# ─────────────────────────────────────────────
def retrain_walk_forward(X:           np.ndarray,
                          y:           np.ndarray,
                          meta:        pd.DataFrame,
                          seq_len:     int,
                          n_features:  int,
                          old_val_ics: list = None) -> tuple:
    """
    Full walk-forward retrain on new feature set.
    Prints IC comparison vs old model if old_val_ics provided.
    """
    quarters = sorted(meta["quarter"].unique())
    models, val_ics = [], []
    best_ic, best_model = -999, None
 
    print(f"\n── Walk-forward retraining on {n_features} features ──")
    print(f"   Quarters: {len(quarters)} | "
          f"Folds: {len(quarters) - 9}")
 
    for i in range(8, len(quarters) - 1):
        train_qs = set(quarters[:i])
        val_q    = quarters[i]
 
        tr_mask  = meta["quarter"].isin(train_qs)
        val_mask = meta["quarter"] == val_q
 
        X_tr,  y_tr  = X[tr_mask],  y[tr_mask]
        X_val, y_val = X[val_mask], y[val_mask]
 
        if len(X_tr) == 0 or len(X_val) == 0:
            continue
 
        model = build_model(seq_len, n_features)
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=150,
            batch_size=32,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=15,
                              restore_best_weights=True),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=7, min_lr=1e-5, verbose=0)
            ],
            verbose=0
        )
 
        val_pred = model.predict(X_val, verbose=0).flatten()
        ic, _    = spearmanr(y_val, val_pred)
        val_ics.append(ic)
        models.append(model)
 
        # Track best model
        if ic > best_ic:
            best_ic    = ic
            best_model = model
            model.save("best_retrained_model.keras")
 
        # Compare vs old model if available
        old_ic_str = ""
        if old_val_ics and (i - 8) < len(old_val_ics):
            old_ic     = old_val_ics[i - 8]
            delta      = ic - old_ic
            old_ic_str = f" | old={old_ic:+.4f} | delta={delta:+.4f}"
 
        print(f"  Fold {i-7:2d} | {val_q} | "
              f"train={len(X_tr):4d} | val={len(X_val):3d} | "
              f"IC={ic:+.4f}{old_ic_str}")
 
    # ── Summary ───────────────────────────────────────────────────────
    print(f"\n── Retraining complete ──")
    print(f"  New model  avg IC (all folds)    : {np.mean(val_ics):+.4f}")
    print(f"  New model  avg IC (last 4 folds) : {np.mean(val_ics[-4:]):+.4f}")
    print(f"  Best single fold IC              : {best_ic:+.4f}")
 
    if old_val_ics:
        n = min(len(val_ics), len(old_val_ics))
        improvement = np.mean(val_ics[:n]) - np.mean(old_val_ics[:n])
        print(f"\n  Old model avg IC : {np.mean(old_val_ics[:n]):+.4f}")
        print(f"  New model avg IC : {np.mean(val_ics[:n]):+.4f}")
        print(f"  Improvement      : {improvement:+.4f}")
        if improvement > 0.05:
            print("  → Significant improvement — use new model")
        elif improvement > 0:
            print("  → Marginal improvement — new model slightly better")
        else:
            print("  → No improvement — consider keeping old model or blending")
 
    return models, val_ics, best_model
 
 
# ─────────────────────────────────────────────
# STEP 5: PREDICT AND RANK TODAY
# ─────────────────────────────────────────────
def predict_and_rank_today(X_today:     np.ndarray,
                            meta_today:  pd.DataFrame,
                            models:      list) -> pd.DataFrame:
    """
    Ensemble prediction on today's sequences → ranked output.
    """
    preds = np.mean(
        [m.predict(X_today, verbose=0).flatten() for m in models],
        axis=0
    )
 
    results = meta_today.copy()
    results["raw_score"]  = preds
    results["rank_pct"]   = results["raw_score"].rank(pct=True)
    results["rank_int"]   = results["raw_score"].rank(
                                ascending=False, method="min").astype(int)
    results["top20pct"]   = (results["rank_pct"] >= 0.80).astype(int)
    results["signal"]     = results["rank_pct"].apply(
        lambda x: "STRONG BUY" if x >= 0.90 else
                  "BUY"        if x >= 0.80 else
                  "WATCH"      if x >= 0.65 else
                  "NEUTRAL"    if x >= 0.35 else
                  "AVOID"
    )
 
    results = results.sort_values("rank_int").reset_index(drop=True)
 
    print(f"\n── Today's rankings (as of {results['as_of'].iloc[0]}) ──")
    print(f"\n  {'Rank':<6} {'Ticker':<10} {'Score':>8} "
          f"{'Percentile':>12} {'Signal':<12}")
    print(f"  {'─'*6} {'─'*10} {'─'*8} {'─'*12} {'─'*12}")
    for _, row in results.iterrows():
        marker = " ◀" if row["top20pct"] == 1 else ""
        print(f"  {int(row['rank_int']):<6} {row['ticker']:<10} "
              f"{row['raw_score']:>8.4f} "
              f"{row['rank_pct']:>11.1%} "
              f"{row['signal']:<12}{marker}")
 
    return results


In [34]:
# ─────────────────────────────────────────────
# USAGE IN YOUR NOTEBOOK
# ─────────────────────────────────────────────
"""
# Prerequisites:
#   models      : trained LSTM ensemble from your walk-forward loop
#   all_pred_df : quarterly predictions from your dual LSTM
"""
all_pred_df = pd.read_parquet("all_stock_predictions.parquet")
print(all_pred_df.head())

tickers = all_pred_df["ticker"].unique().tolist()
print( len(tickers)) 
# 1. Fetch prices up to today
prices = fetch_prices_to_today(tickers, start="2018-01-01")

# 2. Full technical features
prices = compute_technical_features(prices)

# 3. Weekly cross-sectional rank
prices = add_weekly_rank(prices)

# 4. Attach predicted rank history
prices = attach_predicted_rank(prices, all_pred_df)




  ticker quarter  pred_rank
0   AAPL  2025Q4   0.595355
1   ABNB  2025Q4   0.600774
2   ADBE  2025Q4   0.607504
3    ADI  2026Q1   0.631528
4    ADP  2025Q4   0.605950
83
Fetching weekly prices: 2018-01-01 → 2026-03-24
  Done: 83 tickers | 35,336 rows | latest: 2026-03-23
  Technical features computed: 48 indicators


In [29]:
# 5. Define features
feature_cols = [
    # Returns
    "ret_1w","ret_2w","ret_4w","ret_8w","ret_13w","ret_26w","ret_52w",
    "mom_12m_skip1m",
    # Trend
    "ema_cross_8_26","ema_cross_13_52","price_vs_ema26","price_vs_ema52",
    "macd","macd_hist","macd_cross","price_vs_52w_ma",
    # Momentum
    "rsi_14w","rsi_divergence","stoch_k","stoch_d","roc_4w","roc_13w",
    # Volatility
    "vol_4w","vol_13w","vol_52w","vol_regime",
    "bb_position","bb_width","atr_pct","drawdown","pos_52w_range",
    # Sharpe
    "sharpe_13w","sharpe_52w",
    # Volume
    "vol_ratio","vol_surge","obv_norm",
    # Weekly rank
    "weekly_rank_pct","weekly_rank_ma4","weekly_rank_trend",
    "rsi_rank_pct","mom_rank_pct",
    # Predicted rank history
    "pred_rank",
    "pred_rank_lag1","pred_rank_lag2","pred_rank_lag3","pred_rank_lag4",
    "pred_rank_trend","pred_rank_consistency"
]


In [26]:
print ( prices.columns)

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'ret_1w',
       'ret_2w', 'ret_4w', 'ret_8w', 'ret_13w', 'ret_26w', 'ret_52w',
       'mom_12m_skip1m', 'ema_8w', 'ema_13w', 'ema_26w', 'ema_52w',
       'ema_cross_8_26', 'ema_cross_13_52', 'price_vs_ema26', 'price_vs_ema52',
       'macd', 'macd_signal', 'macd_hist', 'macd_cross', 'rsi_14w',
       'rsi_divergence', 'stoch_k', 'stoch_d', 'roc_4w', 'roc_13w', 'vol_4w',
       'vol_13w', 'vol_26w', 'vol_52w', 'vol_regime', 'bb_position',
       'bb_width', 'atr_14w', 'atr_pct', 'sharpe_13w', 'sharpe_52w',
       'high_52w', 'drawdown', 'low_52w', 'pos_52w_range', 'ma_13w', 'ma_52w',
       'price_vs_52w_ma', 'vol_ratio', 'vol_surge', 'obv_norm', 'vwap_ratio',
       'weekly_rank_pct', 'rsi_rank_pct', 'mom_rank_pct', 'weekly_rank_ma4',
       'weekly_rank_trend', 'quarter', 'pred_rank', 'pred_rank_lag1',
       'pred_rank_lag2', 'pred_rank_lag3', 'pred_rank_lag4', 'pred_rank_trend',
       'pred_rank_consistency'],
    

In [27]:
# ─────────────────────────────────────────────
# USAGE — paste into your notebook
# ─────────────────────────────────────────────
"""
# Prerequisites already done:
#   prices       : from fetch_prices_to_today() + compute_technical_features()
#                  + add_weekly_rank() + attach_predicted_rank()
"""
#   old_val_ics  : val_ics from your previous walk-forward loop
#   feature_cols : full list including TA + rank features

tickers = df['ticker'].unique().tolist()
print( len(tickers))

df.isna().sum()[df.isna().sum() > 0]
df = df.dropna(subset=["close_price"])

df.isna().sum()[df.isna().sum() > 0]
df = df.sort_values(["ticker", "fiscalDateEnding"])

df["future_return_1y"] = ( df.groupby("ticker")["close_price"].shift(-4) / df["close_price"] - 1)

df["fiscalDateEnding"]         = pd.to_datetime(df["fiscalDateEnding"])
df["quarter"]                  = df["fiscalDateEnding"].dt.to_period("Q")

SEQ_LEN = 52


96


In [30]:
labeled_df   = df.dropna(subset=["future_return_1y"])

# ── 1. Build labeled sequences on new feature set ─────────────────────
X, y, meta = build_labeled_sequences( prices       = prices,
                                      labeled_df   = labeled_df,
                                      feature_cols = feature_cols,
                                      seq_len      = SEQ_LEN
                                    )


Labeled sequences built:
  X shape  : (1635, 52, 48)  (samples × 52 weeks × 48 features)
  y shape  : (1635,)
  Skipped  : {'no_label': 825, 'short': 254, 'nan': 0}
  Quarters : ['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1']


In [37]:

# ── 2. Build today's sequences ────────────────────────────────────────
X_today, meta_today = build_today_sequences(prices       = prices,
                                            feature_cols = feature_cols,
                                             seq_len      = SEQ_LEN
                                           )
 
# ── 3. Retrain walk-forward ───────────────────────────────────────────
new_models, new_val_ics, best_model = retrain_walk_forward(  X            = X,
                                                             y            = y,
                                                             meta         = meta,
                                                             seq_len      = SEQ_LEN,
                                                             n_features   = len(feature_cols),
                                                             old_val_ics  = None     # your old model's val ICs for comparison
                                                            )
 
# ── 4. Predict today ──────────────────────────────────────────────────
today_results = predict_and_rank_today(X_today, meta_today, new_models)
 
# ── 5. Technical summary for top picks ───────────────────────────────
top_tickers = today_results[today_results["top20pct"] == 1]["ticker"].tolist()
ta_summary  = technical_summary(prices, top_tickers)
print(ta_summary.to_string(index=False))
 
# ── 6. Save models ────────────────────────────────────────────────────
import pickle
with open("new_models.pkl", "wb") as f:
    pickle.dump(new_models, f)
print("Models saved.")




Today sequences: (83, 52, 48)  (83 tickers as of today)

── Walk-forward retraining on 48 features ──
   Quarters: 21 | Folds: 12
  Fold  1 | 2022Q1 | train= 634 | val= 83 | IC=+0.0457
  Fold  2 | 2022Q2 | train= 717 | val= 83 | IC=-0.0092
  Fold  3 | 2022Q3 | train= 800 | val= 83 | IC=+0.0708
  Fold  4 | 2022Q4 | train= 883 | val= 83 | IC=+0.1956
  Fold  5 | 2023Q1 | train= 966 | val= 83 | IC=+0.0736
  Fold  6 | 2023Q2 | train=1049 | val= 83 | IC=+0.0128
  Fold  7 | 2023Q3 | train=1132 | val= 83 | IC=+0.1371
  Fold  8 | 2023Q4 | train=1215 | val= 83 | IC=+0.2062
  Fold  9 | 2024Q1 | train=1298 | val= 83 | IC=+0.3232
  Fold 10 | 2024Q2 | train=1381 | val= 83 | IC=+0.3227
  Fold 11 | 2024Q3 | train=1464 | val= 83 | IC=+0.3164
  Fold 12 | 2024Q4 | train=1547 | val= 80 | IC=-0.0586

── Retraining complete ──
  New model  avg IC (all folds)    : +0.1364
  New model  avg IC (last 4 folds) : +0.2259
  Best single fold IC              : +0.3232

── Today's rankings (as of 2026-03-23) ──

  R